# Labo III
# Multinacional - Prediccion de Ventas

## Importamos librerias

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# Cargar datos
print("Cargando datos...")
data = pd.read_csv("../datasets/sell-in.txt", sep="\t")
products_to_predict = pd.read_csv('../datasets/product_id_apredecir201912.txt')

# Preparar dataset
print("Preparando dataset...")
df = data.groupby(['product_id', 'periodo'])['tn'].sum().reset_index()

# Crear features (lags) y target (t+2)
print("Creando features...")
df['target'] = df.groupby('product_id')['tn'].shift(-2)  # t+2 (Feb 2020)
for i in range(1, 12):
    df[f'tn_{i}'] = df.groupby('product_id')['tn'].shift(i)

# Productos mágicos para entrenar
magicos = [20002, 20003, 20006, 20010, 20011, 20018, 20019, 20021, 20026, 20028, 
           20035, 20039, 20042, 20044, 20045, 20046, 20049, 20051, 20052, 20053, 
           20055, 20008, 20001, 20017, 20086, 20180, 20193, 20320, 20532, 20612, 
           20637, 20807, 20838]

# Entrenar modelo con datos de diciembre 2018
print("Entrenando modelo...")
train_data = df[(df['periodo'] == 201812) & (df['product_id'].isin(magicos))].dropna()
features = ['tn'] + [f'tn_{i}' for i in range(1, 12)]
X_train = train_data[features]
y_train = train_data['target']

model = LinearRegression()
model.fit(X_train, y_train)

print(f"Modelo entrenado con {len(train_data)} registros")
print(f"Coeficientes: {model.coef_[:3]}...")  # Primeros 3 coeficientes

# Generar predicciones para febrero 2020
print("Generando predicciones...")
predicciones = []

# Datos de diciembre 2019 para hacer predicciones
dec_2019 = df[df['periodo'] == 201912].copy()

# Calcular promedio histórico (últimos 12 meses excluyendo ceros)
hist_data = df[df['periodo'].between(201812, 201911)]
promedios = hist_data[hist_data['tn'] > 0].groupby('product_id')['tn'].mean()

# Para cada producto a predecir
for product_id in products_to_predict['product_id']:
    
    # Si está en productos mágicos Y tiene datos completos, usar modelo
    if product_id in magicos:
        product_data = dec_2019[dec_2019['product_id'] == product_id]
        
        if not product_data.empty and not product_data[features].isnull().any().any():
            # Usar modelo de regresión
            X_pred = product_data[features]
            prediccion = model.predict(X_pred)[0]
            predicciones.append({'product_id': product_id, 'tn': max(0, prediccion)})
            continue
    
    # Si no, usar promedio histórico
    if product_id in promedios:
        prediccion = promedios[product_id]
    else:
        prediccion = 0  # Si no hay datos históricos
    
    predicciones.append({'product_id': product_id, 'tn': prediccion})

# Crear DataFrame final
submission = pd.DataFrame(predicciones)

# Verificar que tenemos todos los productos
print(f"Productos a predecir: {len(products_to_predict)}")
print(f"Predicciones generadas: {len(submission)}")
print(f"Productos con predicción > 0: {(submission['tn'] > 0).sum()}")

# Guardar submission
submission.to_csv('submission_kaggle.csv', index=False)
print("✅ Archivo 'submission_kaggle.csv' guardado exitosamente")

# Mostrar estadísticas
print(f"\nEstadísticas finales:")
print(f"- Predicciones con modelo: {len([p for p in predicciones if products_to_predict.loc[products_to_predict['product_id'] == p['product_id']].index[0] < len(magicos) if p['product_id'] in magicos])}")
print(f"- Predicciones con promedio: {len(submission) - len([p for p in predicciones if p['product_id'] in magicos])}")
print(f"- Valor promedio predicho: {submission['tn'].mean():.4f}")
print(f"- Valor máximo predicho: {submission['tn'].max():.4f}")

submission.head(10)

Cargando datos...
Preparando dataset...
Creando features...
Entrenando modelo...
Modelo entrenado con 33 registros
Coeficientes: [-0.00133878  0.23655828  0.17820788]...
Generando predicciones...
Productos a predecir: 780
Predicciones generadas: 780
Productos con predicción > 0: 780
✅ Archivo 'submission_kaggle.csv' guardado exitosamente

Estadísticas finales:
- Predicciones con modelo: 13
- Predicciones con promedio: 747
- Valor promedio predicho: 38.5993
- Valor máximo predicho: 1183.6406


,product_id,tn
0,20001,1162.707525
1,20002,1183.640604
2,20003,684.763931
3,20004,622.854058
4,20005,649.885925
5,20006,482.886867
6,20007,431.754032
7,20008,422.340199
8,20009,546.342627
9,20010,418.689888
